In [9]:
import pandas as pd
import geopandas as gpd
import numpy as np

In [ ]:
AGR = pd.read_csv('1600501_OIAPOQUE/Agregados_por_setores_demografia_BR.csv', sep = ';')
AGR

C:\Users\carlo\AppData\Local\Temp\ipykernel_40896\2967640360.py:1: DtypeWarning: Columns (1,2,3,4,9,18,25,32,33,34) have mixed types. Specify dtype option on import or set low_memory=False.
  AGR = pd.read_csv('1600501_OIAPOQUE/Agregados_por_setores_demografia_BR.csv', sep = ';')


,V01006,V01007,V01008,V01009,V01010,V01011,V01012,V01013,V01014,V01015,...,V01033,V01034,V01035,V01036,V01037,V01038,V01039,V01040,V01041,COD_setor
0,928,428,500,30,39,24,44,36,25,63,...,62,88,68,58,144,129,124,66,53,110001505000002
1,556,270,286,15,20,20,19,30,20,46,...,47,38,47,44,83,95,48,47,34,110001505000003
2,222,108,114,5,4,9,14,7,5,16,...,15,26,11,11,33,37,25,23,17,110001505000004
3,785,408,377,36,33,34,41,42,27,65,...,61,75,71,53,124,92,88,57,40,110001505000006
4,748,373,375,28,25,33,27,24,31,52,...,54,52,51,61,110,116,93,63,46,110001505000007
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
458767,36,17,19,3,0,0,0,0,3,3,...,3,3,0,4,7,4,4,0,0,530010805440139
458768,633,290,343,18,18,31,18,27,29,36,...,51,38,66,59,89,96,66,54,35,530010805440140
458769,387,181,206,12,14,12,17,10,19,27,...,26,24,22,37,63,57,51,26,29,530010805440141
458770,348,170,178,9,12,13,20,17,11,24,...,33,44,31,21,55,66,39,24,3,530010805440142


In [16]:
# Sexo masculino, 0 a 4 anos     M0a
# Sexo masculino, 5 a 9 anos     M0b
# Sexo masculino, 10 a 14 anos   M1a
# Sexo masculino, 15 a 19 anos   M1b
# Sexo masculino, 20 a 24 anos   M2a
# Sexo masculino, 25 a 29 anos   M2b
# Sexo masculino, 30 a 39 anos   M3
# Sexo masculino, 40 a 49 anos   M4
# Sexo masculino, 50 a 59 anos   M5
# Sexo masculino, 60 a 69 anos   M6
# Sexo masculino, 70 anos ou mais M7
# Sexo feminino, 0 a 4 anos      F0a
# Sexo feminino, 5 a 9 anos      F0b
# Sexo feminino, 10 a 14 anos    F1a
# Sexo feminino, 15 a 19 anos    F1b
# Sexo feminino, 20 a 24 anos    F2a
# Sexo feminino, 25 a 29 anos    F2b
# Sexo feminino, 30 a 39 anos    F3
# Sexo feminino, 40 a 49 anos    F4
# Sexo feminino, 50 a 59 anos    F5
# Sexo feminino, 60 a 69 anos    F6
# Sexo feminino, 70 anos ou mais F7

#AGR_si = AGR[['COD_setor', 'V01009','V01010','V01011','V01012','V01013','V01014','V01015','V01016','V01017','V01018','V01019','V01020','V01021','V01022','V01023',
#     'V01024','V01025','V01026','V01027','V01028','V01029','V01030']]
AGR_si = AGR.rename(columns = {'V01006':'T','V01007':'M','V01008':'F','V01009':'M0a','V01010':'M0b','V01011':'M1a','V01012':'M1b',
    'V01013':'M2a', 'V01014':'M2b','V01015':'M3','V01016':'M4','V01017':'M5','V01018':'M6','V01019':'M7','V01020':'F0a','V01021':'F0b','V01022':'F1a',
    'V01023':'F1b','V01024':'F2a','V01025':'F2b','V01026':'F3','V01027':'F4','V01028':'F5','V01029':'F6','V01030':'F7',
    'V01031':'a0a','V01032':'a0b','V01033':'a1a','V01034':'a1b', 'V01035':'a2a','V01036':'a2b','V01037':'a3','V01038':'a4','V01039':'a5',
    'V01040':'a6','V01041':'a7'})

AGR_si = AGR_si[AGR_si['T'] != 'X'].reset_index(drop=True)
AGR_si['M'] = AGR_si['M'].mask((AGR_si['M'] == 'X') & (AGR_si['F'] != 'X'), 
                               AGR_si['T'].astype('int') - AGR_si['F'].replace('X','0').astype('int'))
AGR_si['F'] = AGR_si['F'].mask((AGR_si['F'] == 'X') & (AGR_si['M'] != 'X'), 
                               AGR_si['T'].astype('int') - AGR_si['M'].replace('X','0').astype('int'))
for a in ['0a','0b','1a','1b','2a','2b','3','4','5','6','7']:
    for b in ['M','F']:
        c = 'M' if b == 'F' else 'F'
        AGR_si[f'{b}{a}'] = AGR_si[f'{b}{a}'].mask((AGR_si[f'{b}{a}'] == 'X') & 
                                                    (AGR_si[f'{c}{a}'] != 'X')&
                                                    (AGR_si[f'a{a}'] != 'X'),
                                                    AGR_si[f'a{a}'].replace('X','0').astype('int') - 
                                                    AGR_si[f'{c}{a}'].replace('X','0').astype('int'))
    AGR_si[f'a{a}'] = AGR_si[f'a{a}'].mask((AGR_si[f'a{a}'] == 'X') & 
                                                    (AGR_si[f'M{a}'] != 'X')&
                                                    (AGR_si[f'F{a}'] != 'X'),
                                                    AGR_si[f'M{a}'].replace('X','0').astype('int') +
                                                    AGR_si[f'F{a}'].replace('X','0').astype('int'))

In [6]:
AGR_si[AGR_si.isin(['X']).any(axis=1)]

,CD_setor,T,M,F,M0a,M0b,M1a,M1b,M2a,M2b,...,a0b,a1a,a1b,a2a,a2b,a3,a4,a5,a6,a7
13,110001505000021,61,39,22,3,5,3,X,4,3,...,6,4,3,6,6,6,11,9,X,4
15,110001505000026,153,74,79,X,7,5,5,5,7,...,13,7,12,11,12,21,17,29,10,8
17,110001505000028,229,122,107,8,9,10,9,11,8,...,18,19,15,16,18,37,40,33,10,5
18,110001505000029,56,34,22,3,3,3,0,3,X,...,4,4,X,7,3,12,8,6,6,X
19,110001505000030,43,27,16,3,X,3,X,X,X,...,X,4,X,X,3,9,10,4,X,X
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450072,530010805440128,20,12,8,0,X,0,X,X,0,...,X,X,X,X,X,3,3,6,0,0
450077,530010805440133,496,238,258,13,24,22,22,27,16,...,46,45,46,46,37,84,80,56,16,8
450079,530010805440135,129,59,70,4,6,3,X,4,3,...,16,10,7,5,10,29,10,12,15,6
450081,530010805440137,149,85,64,6,4,5,14,3,5,...,7,9,18,9,8,19,29,20,14,5


In [31]:
AGR_si[AGR_si.isna().any(axis=1)]

,CD_setor,T,M,F,M0a,M0b,M1a,M1b,M2a,M2b,...,a0b,a1a,a1b,a2a,a2b,a3,a4,a5,a6,a7
13,110001505000021,61.0,39.0,22.0,3.0,5.0,3.0,2.0,4.0,3.0,...,6.0,4.0,3.0,6.0,6.0,6.0,11.0,9.0,NaN,4.0
18,110001505000029,56.0,34.0,22.0,3.0,3.0,3.0,0.0,3.0,1.0,...,4.0,4.0,NaN,7.0,3.0,12.0,8.0,6.0,6.0,NaN
19,110001505000030,43.0,27.0,16.0,3.0,3.0,3.0,2.0,1.0,1.0,...,NaN,4.0,NaN,NaN,3.0,9.0,10.0,4.0,NaN,NaN
23,110001505000051,15.0,10.0,5.0,0.0,0.0,1.0,2.0,0.0,0.0,...,0.0,NaN,NaN,0.0,0.0,NaN,NaN,7.0,NaN,NaN
26,110001505000054,92.0,46.0,46.0,3.0,3.0,3.0,7.0,6.0,2.0,...,7.0,7.0,8.0,11.0,4.0,9.0,17.0,10.0,8.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450055,530010805440109,66.0,38.0,28.0,4.0,4.0,2.0,2.0,1.0,3.0,...,5.0,3.0,5.0,5.0,5.0,11.0,6.0,14.0,6.0,NaN
450057,530010805440112,40.0,20.0,20.0,0.0,3.0,1.0,1.0,1.0,4.0,...,4.0,NaN,NaN,4.0,6.0,5.0,8.0,4.0,NaN,NaN
450066,530010805440121,300.0,142.0,158.0,11.0,8.0,20.0,8.0,10.0,10.0,...,25.0,32.0,19.0,20.0,23.0,59.0,64.0,28.0,12.0,4.0
450072,530010805440128,20.0,12.0,8.0,0.0,1.0,0.0,1.0,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,3.0,3.0,6.0,0.0,0.0


In [ ]:
def preencher_nan_inteiros(row, colunas_alvo,total):
    conhecidos = row[colunas_alvo].dropna()
    soma_conhecidos = conhecidos.sum()
    total_necessario = row[total]
    
    colunas_nan = row[colunas_alvo][row[colunas_alvo].isna()].index.tolist()
    num_nan = len(colunas_nan)
    
    if num_nan == 0:
        return row
    
    valor_restante = total_necessario - soma_conhecidos
    
    if valor_restante < 0:
        raise ValueError(f"Valor total M ({total_necessario}) é menor que soma dos conhecidos ({soma_conhecidos})")
    
    # Gerar inteiros aleatórios positivos que somam valor_restante
    # Usamos multinomial (que requer valor_restante inteiro)
    if valor_restante > 0:
        # Multinomial: n=valor_restante, pvals uniformes
        pvals = np.ones(num_nan) / num_nan
        distribuicao = np.random.multinomial(int(valor_restante-num_nan), pvals) + 1
    else:
        distribuicao = np.zeros(num_nan, dtype=int)
    
    # Atribuir (garantindo pelo menos 1 se valor_restante >= num_nan)
    for col, val in zip(colunas_nan, distribuicao):
        row[col] = val
    
    return row

In [23]:
AGR_si.replace('X', np.nan, inplace=True)
AGR_si.iloc[:,1:]= AGR_si.iloc[:,1:].astype('float64')
AGR_si

,CD_setor,T,M,F,M0a,M0b,M1a,M1b,M2a,M2b,...,a0b,a1a,a1b,a2a,a2b,a3,a4,a5,a6,a7
0,110001505000002,928.0,428.0,500.0,30.0,39.0,24.0,44.0,36.0,25.0,...,68.0,62.0,88.0,68.0,58.0,144.0,129.0,124.0,66.0,53.0
1,110001505000003,556.0,270.0,286.0,15.0,20.0,20.0,19.0,30.0,20.0,...,36.0,47.0,38.0,47.0,44.0,83.0,95.0,48.0,47.0,34.0
2,110001505000004,222.0,108.0,114.0,5.0,4.0,9.0,14.0,7.0,5.0,...,11.0,15.0,26.0,11.0,11.0,33.0,37.0,25.0,23.0,17.0
3,110001505000006,785.0,408.0,377.0,36.0,33.0,34.0,41.0,42.0,27.0,...,63.0,61.0,75.0,71.0,53.0,124.0,92.0,88.0,57.0,40.0
4,110001505000007,748.0,373.0,375.0,28.0,25.0,33.0,27.0,24.0,31.0,...,52.0,54.0,52.0,51.0,61.0,110.0,116.0,93.0,63.0,46.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450083,530010805440139,36.0,17.0,19.0,3.0,NaN,NaN,0.0,0.0,3.0,...,3.0,3.0,3.0,NaN,4.0,7.0,4.0,4.0,NaN,NaN
450084,530010805440140,633.0,290.0,343.0,18.0,18.0,31.0,18.0,27.0,29.0,...,41.0,51.0,38.0,66.0,59.0,89.0,96.0,66.0,54.0,35.0
450085,530010805440141,387.0,181.0,206.0,12.0,14.0,12.0,17.0,10.0,19.0,...,27.0,26.0,24.0,22.0,37.0,63.0,57.0,51.0,26.0,29.0
450086,530010805440142,348.0,170.0,178.0,9.0,12.0,13.0,20.0,17.0,11.0,...,19.0,33.0,44.0,31.0,21.0,55.0,66.0,39.0,24.0,3.0


In [ ]:
colunas_idade_masculina = ['M0a','M0b','M1a','M1b','M2a','M2b','M3','M4','M5','M6','M7']
AGR_si[AGR_si[colunas_idade_masculina].isna().any(axis=1)] = AGR_si[AGR_si[colunas_idade_masculina].isna().any(axis=1)].apply(
    lambda row: preencher_nan_inteiros(row, colunas_idade_masculina, 'M'), axis=1)

In [32]:
colunas_idade_feminina = ['F0a','F0b','F1a','F1b','F2a','F2b','F3','F4','F5','F6','F7']
AGR_si[AGR_si[colunas_idade_feminina].isna().any(axis=1)] = AGR_si[AGR_si[colunas_idade_feminina].isna().any(axis=1)].apply(
    lambda row: preencher_nan_inteiros(row, colunas_idade_feminina, 'F'), axis=1)

In [38]:
for a in ['0a','0b','1a','1b','2a','2b','3','4','5','6','7']:
    AGR_si[f'a{a}'] = AGR_si[f'a{a}'].mask((AGR_si[f'a{a}'].isna()), AGR_si[f'M{a}'] + AGR_si[f'F{a}'])

AGR_si['COD_setor'] = AGR_si['CD_setor'].astype('str')
AGR_si

,CD_setor,T,M,F,M0a,M0b,M1a,M1b,M2a,M2b,...,a1a,a1b,a2a,a2b,a3,a4,a5,a6,a7,COD_setor
0,110001505000002,928.0,428.0,500.0,30.0,39.0,24.0,44.0,36.0,25.0,...,62.0,88.0,68.0,58.0,144.0,129.0,124.0,66.0,53.0,110001505000002
1,110001505000003,556.0,270.0,286.0,15.0,20.0,20.0,19.0,30.0,20.0,...,47.0,38.0,47.0,44.0,83.0,95.0,48.0,47.0,34.0,110001505000003
2,110001505000004,222.0,108.0,114.0,5.0,4.0,9.0,14.0,7.0,5.0,...,15.0,26.0,11.0,11.0,33.0,37.0,25.0,23.0,17.0,110001505000004
3,110001505000006,785.0,408.0,377.0,36.0,33.0,34.0,41.0,42.0,27.0,...,61.0,75.0,71.0,53.0,124.0,92.0,88.0,57.0,40.0,110001505000006
4,110001505000007,748.0,373.0,375.0,28.0,25.0,33.0,27.0,24.0,31.0,...,54.0,52.0,51.0,61.0,110.0,116.0,93.0,63.0,46.0,110001505000007
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450083,530010805440139,36.0,17.0,19.0,3.0,1.0,3.0,0.0,0.0,3.0,...,3.0,3.0,1.0,4.0,7.0,4.0,4.0,3.0,1.0,530010805440139
450084,530010805440140,633.0,290.0,343.0,18.0,18.0,31.0,18.0,27.0,29.0,...,51.0,38.0,66.0,59.0,89.0,96.0,66.0,54.0,35.0,530010805440140
450085,530010805440141,387.0,181.0,206.0,12.0,14.0,12.0,17.0,10.0,19.0,...,26.0,24.0,22.0,37.0,63.0,57.0,51.0,26.0,29.0,530010805440141
450086,530010805440142,348.0,170.0,178.0,9.0,12.0,13.0,20.0,17.0,11.0,...,33.0,44.0,31.0,21.0,55.0,66.0,39.0,24.0,3.0,530010805440142


In [39]:
AGR_si.to_csv('1600501_OIAPOQUE/Agregados_por_setores_demografia_BR_corrigido.csv', sep=';', index=False)

In [6]:
AGR_si = pd.read_csv('1600501_OIAPOQUE/Agregados_por_setores_demografia_BR_corrigido.csv', sep=';')
AGR_si['COD_setor'] = AGR_si['CD_setor'].astype('str')
AGR_si

,CD_setor,T,M,F,M0a,M0b,M1a,M1b,M2a,M2b,...,a1a,a1b,a2a,a2b,a3,a4,a5,a6,a7,COD_setor
0,110001505000002,928.0,428.0,500.0,30.0,39.0,24.0,44.0,36.0,25.0,...,62.0,88.0,68.0,58.0,144.0,129.0,124.0,66.0,53.0,110001505000002
1,110001505000003,556.0,270.0,286.0,15.0,20.0,20.0,19.0,30.0,20.0,...,47.0,38.0,47.0,44.0,83.0,95.0,48.0,47.0,34.0,110001505000003
2,110001505000004,222.0,108.0,114.0,5.0,4.0,9.0,14.0,7.0,5.0,...,15.0,26.0,11.0,11.0,33.0,37.0,25.0,23.0,17.0,110001505000004
3,110001505000006,785.0,408.0,377.0,36.0,33.0,34.0,41.0,42.0,27.0,...,61.0,75.0,71.0,53.0,124.0,92.0,88.0,57.0,40.0,110001505000006
4,110001505000007,748.0,373.0,375.0,28.0,25.0,33.0,27.0,24.0,31.0,...,54.0,52.0,51.0,61.0,110.0,116.0,93.0,63.0,46.0,110001505000007
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450083,530010805440139,36.0,17.0,19.0,3.0,1.0,3.0,0.0,0.0,3.0,...,3.0,3.0,1.0,4.0,7.0,4.0,4.0,3.0,1.0,530010805440139
450084,530010805440140,633.0,290.0,343.0,18.0,18.0,31.0,18.0,27.0,29.0,...,51.0,38.0,66.0,59.0,89.0,96.0,66.0,54.0,35.0,530010805440140
450085,530010805440141,387.0,181.0,206.0,12.0,14.0,12.0,17.0,10.0,19.0,...,26.0,24.0,22.0,37.0,63.0,57.0,51.0,26.0,29.0,530010805440141
450086,530010805440142,348.0,170.0,178.0,9.0,12.0,13.0,20.0,17.0,11.0,...,33.0,44.0,31.0,21.0,55.0,66.0,39.0,24.0,3.0,530010805440142


In [7]:
setores = gpd.read_file('1600501_OIAPOQUE/AP_setores_CD2022.shp')
setores = setores[setores['NM_MUN']=='Oiapoque']

setores_ = setores.merge(AGR_si, left_on='CD_SETOR', right_on = 'COD_setor', how= 'inner')
setores_

,CD_SETOR,SITUACAO,CD_SIT,CD_TIPO,AREA_KM2,CD_REGIAO,NM_REGIAO,CD_UF,NM_UF,CD_MUN,...,a1a,a1b,a2a,a2b,a3,a4,a5,a6,a7,COD_setor
0,160050105000001,Urbana,1,0,0.117806,1,Norte,16,Amapá,1600501,...,77.0,57.0,57.0,62.0,112.0,82.0,88.0,29.0,21.0,160050105000001
1,160050105000004,Urbana,1,0,0.215701,1,Norte,16,Amapá,1600501,...,53.0,81.0,86.0,76.0,114.0,111.0,80.0,54.0,28.0,160050105000004
2,160050105000007,Rural,5,5,0.706869,1,Norte,16,Amapá,1600501,...,4.0,1.0,1.0,3.0,3.0,3.0,1.0,1.0,2.0,160050105000007
3,160050105000008,Rural,8,0,69.623041,1,Norte,16,Amapá,1600501,...,1.0,5.0,1.0,1.0,1.0,4.0,4.0,0.0,1.0,160050105000008
4,160050105000011,Rural,5,5,0.312076,1,Norte,16,Amapá,1600501,...,3.0,5.0,6.0,4.0,5.0,3.0,4.0,2.0,1.0,160050105000011
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,160050115000027,Rural,8,5,0.670449,1,Norte,16,Amapá,1600501,...,4.0,2.0,3.0,1.0,4.0,2.0,1.0,0.0,0.0,160050115000027
76,160050115000029,Rural,8,5,0.395323,1,Norte,16,Amapá,1600501,...,3.0,0.0,1.0,3.0,4.0,1.0,1.0,1.0,0.0,160050115000029
77,160050120000001,Rural,5,0,1.169275,1,Norte,16,Amapá,1600501,...,15.0,15.0,17.0,16.0,35.0,34.0,50.0,20.0,1.0,160050120000001
78,160050120000002,Rural,8,0,79.760938,1,Norte,16,Amapá,1600501,...,2.0,0.0,3.0,4.0,6.0,5.0,1.0,1.0,2.0,160050120000002


In [22]:
listagem_total = []
for a in setores_.index:
    s = setores_.iloc[a]
    cod_setor = s['CD_SETOR']
    listagem = []
    for valor in ['M0a','M0b','M1a','M1b','M2a','M2b','M3','M4','M5','M6','M7',
                'F0a','F0b','F1a','F1b','F2a','F2b','F3','F4','F5','F6','F7']:
        listagem.extend([{'COD_setor': cod_setor, 'sexo': valor[0], 'faixa_etaria': valor[1:], 'tipo_domicilio':np.nan,'id_domicilio':np.nan}]*int(s[valor]))

    listagem = np.random.choice(listagem, size = len(listagem), replace= False)

    listagem_total.extend(listagem)

In [23]:
pd.DataFrame(listagem_total)

,COD_setor,sexo,faixa_etaria,tipo_domicilio,id_domicilio
0,160050105000001,M,3,NaN,NaN
1,160050105000001,M,3,NaN,NaN
2,160050105000001,F,4,NaN,NaN
3,160050105000001,F,0a,NaN,NaN
4,160050105000001,M,6,NaN,NaN
...,...,...,...,...,...
27307,160050120000003,F,3,NaN,NaN
27308,160050120000003,M,5,NaN,NaN
27309,160050120000003,M,2a,NaN,NaN
27310,160050120000003,M,7,NaN,NaN


In [24]:
pd.DataFrame(listagem_total).to_csv('1600501_OIAPOQUE/listagem_individuos_sinteticos.csv', sep=';', index=False)

In [ ]:
## Dá para colocar mais dados, mas tentar, mas por enquanto fica assim